In [2]:
!pip install transformers datasets evaluate rouge_score sentencepiece -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [3]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
import evaluate
import numpy as np

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Seq2SeqLM: Encoder + Decoder + LM head(vocab, 每个位置预测下一个token)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print(model)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [4]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [5]:
print(dataset['train'][0].keys())
print(dataset['train'].column_names)

dict_keys(['article', 'highlights', 'id'])
['article', 'highlights', 'id']


In [6]:
sample = dataset['train'][0]
print(sample['article'][:200])
print(sample['highlights'][:200])
print(f"Article length: {len(sample['article'].split())}")
print(f"Summary length: {len(sample['highlights'].split())}")

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on 
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been hel
Article length: 455
Summary length: 41


In [7]:
print(sample['highlights'][:400])

Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [8]:
sample = dataset['test'][0]
print(sample['article'][:200])
# print(sample['highlights'][:100])

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor


In [9]:
prefix = "summarize: "
max_input_length = 512
max_target_length = 128

def preprocess(examples):
    inputs = [prefix + article for article in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
    )

    # 用 text_target参数来做target
    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

small_train = dataset['train'].select(range(5000))
small_validation = dataset['validation'].select(range(500))

tokenized_train = small_train.map(preprocess, batched=True, remove_columns=small_train.column_names)
tokenzied_validation = small_validation.map(preprocess, batched=True, remove_columns=small_validation.column_names)



Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
print(f"Column names: {tokenized_train.column_names}")
print(f"Input Ids: {tokenized_train[0]['input_ids'][:20]}")
print(f"Labels: {tokenized_train[0]['labels'][:20]}")
print(tokenizer.decode(tokenized_train[0]['input_ids'][:30]))
print(tokenizer.decode(tokenized_train[0]['labels'][:30]))
print(len(tokenized_train[0]['input_ids']))
print(len(tokenized_train[0]['labels']))

Column names: ['input_ids', 'attention_mask', 'labels']
Input Ids: [21603, 10, 301, 24796, 4170, 6, 2789, 41, 18844, 61, 1636, 8929, 16023, 2213, 4173, 6324, 12591, 15, 11391, 592]
Labels: [8929, 16023, 2213, 4173, 6324, 12591, 15, 2347, 3996, 1755, 329, 13462, 38, 3, 88, 5050, 507, 2089, 3, 5]
summarize: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday. Young actor says he has no plans to fri
512
57


In [11]:
rouge = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    return {k: round(v*100, 2) for k, v in result.items()}

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir='./t5-summarization',
    eval_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    logging_steps=100,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenzied_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.140963,2.202398,30.170000,10.960000,21.750000,21.720000
2,1.999442,2.214315,30.590000,11.000000,21.730000,21.740000
3,1.895882,2.228818,30.190000,10.840000,21.350000,21.360000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1875, training_loss=2.025239103190104, metrics={'train_runtime': 715.9786, 'train_samples_per_second': 20.95, 'train_steps_per_second': 2.619, 'total_flos': 2030127022080000.0, 'train_loss': 2.025239103190104, 'epoch': 3.0})

In [12]:
eval_results = trainer.evaluate()
print(eval_results)

from transformers import pipeline

summarizer = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

text = """
Scientists have discovered a new species of deep-sea fish in the
Mariana Trench. The fish, named Pseudoliparis swirei, was found at
a depth of approximately 8,000 meters. Researchers say this is the
deepest fish ever recorded. The discovery was made during a recent
expedition using advanced underwater drones. The team spent three
months exploring the trench and collected numerous samples.
"""



result = summarizer(text, max_new_tokens=50)
print(result[0]["generated_text"])

{'eval_loss': 2.2023978233337402, 'eval_rouge1': 30.17, 'eval_rouge2': 10.96, 'eval_rougeL': 21.75, 'eval_rougeLsum': 21.72, 'eval_runtime': 75.1385, 'eval_samples_per_second': 6.654, 'eval_steps_per_second': 0.838, 'epoch': 3.0}


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [13]:
eval_results = trainer.evaluate()
print(eval_results)

from transformers import pipeline

summarizer = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

text = """
Scientists have discovered a new species of deep-sea fish in the
Mariana Trench. The fish, named Pseudoliparis swirei, was found at
a depth of approximately 8,000 meters. Researchers say this is the
deepest fish ever recorded. The discovery was made during a recent
expedition using advanced underwater drones. The team spent three
months exploring the trench and collected numerous samples.
"""



result = summarizer(text, max_new_tokens=50)
print(result[0]["generated_text"])

KeyboardInterrupt: 

In [14]:
summarizer = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

text = """
summarize: Scientists have discovered a new species of deep-sea fish in the
Mariana Trench. The fish, named Pseudoliparis swirei, was found at
a depth of approximately 8,000 meters. Researchers say this is the
deepest fish ever recorded. The discovery was made during a recent
expedition using advanced underwater drones. The team spent three
months exploring the trench and collected numerous samples.
"""

result = summarizer(text, max_new_tokens=50)
print(result[0]["generated_text"])

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa


summarize: Scientists have discovered a new species of deep-sea fish in the 
Mariana Trench. The fish, named Pseudoliparis swirei, was found at 
a depth of approximately 8,000 meters. Researchers say this is the 
deepest fish ever recorded. The discovery was made during a recent 
expedition using advanced underwater drones. The team spent three 
months exploring the trench and collected numerous samples.



In [15]:
text = """
Scientists have discovered a new species of deep-sea fish in the
Mariana Trench. The fish, named Pseudoliparis swirei, was found at
a depth of approximately 8,000 meters. Researchers say this is the
deepest fish ever recorded. The discovery was made during a recent
expedition using advanced underwater drones. The team spent three
months exploring the trench and collected numerous samples.
"""

inputs = tokenizer("summarize: " + text, return_tensors="pt",
                    max_length=512, truncation=True).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50, num_beams=4)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Pseudoliparis swirei found at a depth of approximately 8,000 meters. Scientists say this is the deepest fish ever recorded.


In [16]:
text = "sumarize: " + sample['article'][:2000]
inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True).to(model.device)

greedy = model.generate(**inputs, max_length=80)
print("Greedy:", tokenizer.decode(greedy[0], skip_special_tokens=True))

beam = model.generate(**inputs, max_length=80, num_beams=4, no_repeat_ngram_size=3)
print("Beam:", tokenizer.decode(beam[0], skip_special_tokens=True))

sample_out = model.generate(**inputs, max_length=80, do_sample=True, top_p=0.9, temperature=0.8)
print("Sample:", tokenizer.decode(sample_out[0],skip_special_tokens=True))

Greedy: Palestinian Authority officially becomes 123rd member of International Criminal Court. The ICC opened preliminary examination into the situation in Palestinian territories. Palestinian Foreign Minister Riad al-Malki: Acceding to the treaty is just the first step for the Palestinians.
Beam: "We are also a step closer to ending a long era of impunity and injustice," ICC official says. Palestinian Authority officially became 123rd member of International Criminal Court on Wednesday. The ICC opened preliminary examination into the situation in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body.
Sample: "We are also a step closer to ending a long era of impunity and injustice," Palestinians say. The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday. The ICC opened a preliminary examination into the situation in Palestinian territories.
